# ⚽💰 Football Finance Intelligence
## Module 2 — Le joueur vu comme un actif financier

**Idée :** un club qui achète un joueur fait un **investissement**. Comme une action, la valeur d'un joueur évolue dans le temps : elle peut monter, baisser, être plus ou moins volatile, et elle se **déprécie** avec l'âge.

### Questions étudiées
1. À quel âge un joueur atteint-il sa **valeur maximale**, et cela dépend-il du poste ?
2. Quel est le **rendement** et le **risque (volatilité)** de la valeur d'un joueur ?
3. Peut-on calculer un **ratio de Sharpe** pour comparer les joueurs comme des actifs ?
4. Comment évolue la **valeur comptable** d'un joueur (amortissement) comparée à sa **valeur de marché** ?

In [2]:
!pip install duckdb -q
!curl -sLO https://pub-e682421888d945d684bcae8890b0ec20.r2.dev/data/transfermarkt-datasets.duckdb

import duckdb
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

con = duckdb.connect("transfermarkt-datasets.duckdb", read_only=True)
def q(sql):
    return con.sql(sql).df()

# Leçon du Module 1 : on regarde la structure avant d'utiliser une table
display(q("DESCRIBE player_valuations"))
display(q("SELECT * FROM player_valuations LIMIT 5"))

,column_name,column_type,null,key,default,extra
0,player_id,INTEGER,YES,None,None,None
1,date,DATE,YES,None,None,None
2,market_value_in_eur,INTEGER,YES,None,None,None
3,current_club_name,VARCHAR,YES,None,None,None
4,current_club_id,INTEGER,YES,None,None,None
5,player_club_domestic_competition_id,VARCHAR,YES,None,None,None


,player_id,date,market_value_in_eur,current_club_name,current_club_id,player_club_domestic_competition_id
0,405973,2000-01-20,150000,Unknown,3057,BE1
1,342216,2001-07-20,100000,Unknown,1241,SC1
2,3132,2003-12-09,400000,Dynamo Kyiv,126,TR1
3,6893,2003-12-15,900000,Galatasaray,984,GB1
4,10,2004-10-04,7000000,SV Werder Bremen,398,IT1


## 1. Les données : le « cours de bourse » des joueurs
La table `player_valuations` contient l'historique des valeurs marchandes estimées par Transfermarkt : **une ligne = la valeur d'un joueur à une date donnée**. C'est l'équivalent d'une série de cours boursiers.

⚠️ **Point d'attention :** les colonnes de club et de championnat ne correspondent pas toujours à la date de l'estimation (exemple : le Werder Bremen associé au championnat italien). Elles ne seront donc **pas utilisées** pour l'analyse historique. On retient uniquement `player_id`, `date` et `market_value_in_eur`.

In [3]:
# 1. Bilan qualité des valorisations
display(q("""
SELECT
    COUNT(*)                         AS nb_valorisations,
    COUNT(DISTINCT player_id)        AS nb_joueurs,
    MIN(date)                        AS date_min,
    MAX(date)                        AS date_max,
    ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT player_id), 1) AS valo_par_joueur,
    SUM(CASE WHEN market_value_in_eur IS NULL OR market_value_in_eur = 0
             THEN 1 ELSE 0 END)      AS valeurs_vides
FROM player_valuations
"""))

# 2. Toutes les colonnes de la table players
print(q("DESCRIBE players")["column_name"].tolist())

,nb_valorisations,nb_joueurs,date_min,date_max,valo_par_joueur,valeurs_vides
0,656301,41528,2000-01-20,2026-06-12,15.8,1.0


['player_id', 'first_name', 'last_name', 'name', 'last_season', 'current_club_id', 'player_code', 'country_of_birth', 'city_of_birth', 'country_of_citizenship', 'date_of_birth', 'sub_position', 'position', 'foot', 'height_in_cm', 'contract_expiration_date', 'agent_name', 'image_url', 'international_caps', 'international_goals', 'current_national_team_id', 'url', 'current_club_domestic_competition_id', 'current_club_name', 'market_value_in_eur', 'highest_market_value_in_eur']


### Bilan qualité
| Indicateur | Valeur |
|---|---|
| Nombre de valorisations | 656 301 |
| Nombre de joueurs | 41 528 |
| Période | 2000 – juin 2026 |
| Valorisations par joueur (moyenne) | 15,8 |
| Valeurs vides | 1 |

**Interprétation :** chaque joueur dispose en moyenne d'environ 16 estimations dans le temps, ce qui permet de suivre l'évolution de sa valeur comme un cours boursier. La qualité est excellente : une seule valeur manquante.

La table `players` fournit la **date de naissance** (pour calculer l'âge à chaque estimation) et le **poste** du joueur.

## 2. Le cycle de vie de la valeur : à quel âge un joueur vaut-il le plus ?
**Méthode :** pour chaque estimation, on calcule l'âge du joueur à la date de l'estimation. On calcule ensuite la **valeur médiane** par âge et par poste.

**Pourquoi la médiane ?** Quelques stars valent plus de 100 M€ et feraient exploser la moyenne. La médiane représente le joueur « typique ».

**Période :** estimations depuis juillet 2021, pour travailler sur un marché récent et comparable.

In [4]:
age_valeur = q("""
SELECT
    CAST(FLOOR(DATE_DIFF('day', TRY_CAST(p.date_of_birth AS DATE), v.date) / 365.25) AS INT) AS age,
    p.position,
    COUNT(*)                                          AS nb,
    CAST(MEDIAN(v.market_value_in_eur) / 1e6 AS DOUBLE) AS valeur_mediane_M
FROM player_valuations v
JOIN players p ON v.player_id = p.player_id
WHERE v.date >= DATE '2021-07-01'
  AND v.market_value_in_eur > 0
  AND p.date_of_birth IS NOT NULL
  AND p.position IS NOT NULL
  AND p.position <> 'Missing'
GROUP BY 1, 2
HAVING age BETWEEN 16 AND 38
ORDER BY age
""")

fig = px.line(age_valeur, x="age", y="valeur_mediane_M", color="position",
              markers=True,
              title="Valeur médiane d'un joueur selon son âge et son poste (depuis 2021)",
              labels={"age": "Âge", "valeur_mediane_M": "Valeur médiane (M€)",
                      "position": "Poste"})
fig.update_layout(hovermode="x unified")
fig.show()

# Âge du pic de valeur pour chaque poste
pic = age_valeur.loc[age_valeur.groupby("position")["valeur_mediane_M"].idxmax(),
                     ["position", "age", "valeur_mediane_M", "nb"]]
print("Âge où la valeur médiane est maximale :")
print(pic.round(2).to_string(index=False))

Âge où la valeur médiane est maximale :
  position  age  valeur_mediane_M   nb
    Attack   25              0.80 5370
  Defender   25              0.70 6388
Goalkeeper   30              0.45 1294
  Midfield   25              0.80 5413


### Lecture : le cycle de vie de la valeur d'un joueur
| Phase | Âge | Évolution (attaquant) | Interprétation |
|---|---|---|---|
| **Croissance** | 16 → 25 ans | 0,15 → 0,80 M€ (×5, environ +20 %/an) | L'actif s'apprécie |
| **Plateau** | 25 → 29 ans | environ 0,80 M€ | Valeur maximale et stable |
| **Déclin** | 30 ans et plus | 0,80 → 0,35 M€ entre 29 et 33 ans (environ −18 %/an) | Dépréciation rapide |

- **Joueurs de champ :** pic de valeur entre **25 et 29 ans** (attaquants et milieux à 0,80 M€, défenseurs à 0,70 M€).
- **Gardiens :** valeur environ **2 fois plus faible**, mais pic **plus tardif (30 ans)** et déclin plus lent.
- Les valeurs médianes sont modestes car la base couvre de nombreux championnats. La médiane représente le joueur **typique**, pas les stars.
- *Effet de survie :* après 35 ans, seuls les meilleurs joueurs restent actifs, ce qui maintient artificiellement la médiane.

💼 **Implication pour un club :** acheter vers **21-23 ans** (phase d'appréciation) et revendre vers **28-29 ans** (avant la dépréciation) maximise la valeur de l'investissement.

## 3. Le rendement annuel de l'actif « joueur »
**Méthode :**
1. Pour chaque joueur et chaque année, on retient sa **dernière valeur** de l'année.
2. On calcule le **rendement annuel** : (valeur de l'année ÷ valeur de l'année précédente) − 1.
3. On regroupe par âge : **rendement médian**, et **% de joueurs dont la valeur augmente**.

Contrairement à la courbe précédente, on suit ici **les mêmes joueurs d'une année à l'autre**. Période : 2016 – 2025.

In [5]:
rend = q("""
WITH annuel AS (
    SELECT player_id,
           YEAR(date) AS annee,
           arg_max(market_value_in_eur, date) AS valeur      -- dernière valeur de l'année
    FROM player_valuations
    WHERE market_value_in_eur > 0
    GROUP BY 1, 2
),
avec_prec AS (
    SELECT *,
           LAG(valeur) OVER (PARTITION BY player_id ORDER BY annee) AS valeur_prec,
           LAG(annee)  OVER (PARTITION BY player_id ORDER BY annee) AS annee_prec
    FROM annuel
)
SELECT a.player_id, a.annee, a.valeur, a.valeur_prec,
       a.valeur * 1.0 / a.valeur_prec - 1                   AS rendement,
       a.annee - YEAR(TRY_CAST(p.date_of_birth AS DATE))    AS age,
       p.position
FROM avec_prec a
JOIN players p ON a.player_id = p.player_id
WHERE a.valeur_prec IS NOT NULL
  AND a.annee = a.annee_prec + 1        -- années consécutives uniquement
  AND a.annee BETWEEN 2016 AND 2025
  AND p.date_of_birth IS NOT NULL
""")

r_age = (rend[(rend["age"] >= 17) & (rend["age"] <= 36)]
         .groupby("age")["rendement"]
         .agg(rendement_median="median",
              pct_hausse=lambda s: (s > 0).mean() * 100,
              nb="count")
         .reset_index())
r_age["rendement_median"] *= 100
r_age["sens"] = np.where(r_age["rendement_median"] >= 0, "Appréciation", "Dépréciation")

fig = px.bar(r_age, x="age", y="rendement_median", color="sens",
             color_discrete_map={"Appréciation": "#00CC96", "Dépréciation": "#EF553B"},
             text=r_age["rendement_median"].round(0).astype(int).astype(str) + "%",
             title="Rendement annuel médian de la valeur d'un joueur selon son âge (2016-2025)",
             labels={"age": "Âge", "rendement_median": "Rendement médian (%)"})
fig.show()

print(r_age.round(1).to_string(index=False))

 age  rendement_median  pct_hausse    nb         sens
  17             100.0        68.4   500 Appréciation
  18              66.7        66.5  2460 Appréciation
  19              50.0        62.8  6214 Appréciation
  20              38.5        59.9 11384 Appréciation
  21              25.0        56.5 13988 Appréciation
  22              20.0        54.6 15101 Appréciation
  23               7.1        50.5 15485 Appréciation
  24               0.0        47.9 15410 Appréciation
  25               0.0        45.5 14925 Appréciation
  26               0.0        41.8 14062 Appréciation
  27               0.0        38.2 13127 Appréciation
  28               0.0        34.5 12157 Appréciation
  29              -9.1        28.9 10928 Dépréciation
  30             -16.7        21.6  9753 Dépréciation
  31             -20.0        14.5  8527 Dépréciation
  32             -25.0         9.6  7260 Dépréciation
  33             -28.0         6.0  5959 Dépréciation
  34             -30.0      

### Lecture
| Âge | Rendement médian | % de joueurs en hausse | Phase |
|---|---|---|---|
| 17 ans | +100 % | 68 % | Forte appréciation |
| 20 ans | +38 % | 60 % | Appréciation |
| 23 ans | +7 % | 50 % | **Point de bascule** |
| 24-28 ans | 0 % | 48 % → 35 % | Plateau qui s'érode |
| 30 ans | −17 % | 22 % | Dépréciation |
| 35 ans | −33 % | 3 % | Forte dépréciation |

- **Le point de bascule se situe à 23-24 ans** : à partir de 24 ans, moins d'un joueur sur deux prend de la valeur.
- **Le plateau (24-28 ans) est trompeur** : la valeur médiane ne bouge pas, mais la part de joueurs en hausse diminue chaque année.
- **Après 29 ans, la dépréciation accélère**, jusqu'à −33 % par an à 35 ans.
- *Note technique :* les rendements « ronds » (100 %, 50 %, 33,3 %) reflètent les **paliers** utilisés par Transfermarkt pour estimer les valeurs.

### Analyse transversale ou longitudinale ?
La courbe de la section 2 plaçait le pic entre 25 et 29 ans. Elle comparait des **joueurs différents** à chaque âge, et les meilleurs restent plus longtemps dans les données (effet de sélection). En suivant **les mêmes joueurs** d'une année sur l'autre, on constate que la valeur cesse de croître **dès 23-24 ans**.

💼 **Recommandation affinée :** acheter **avant 22 ans** et revendre vers **27-28 ans**.

## 4. Rendement et risque : le joueur comme un actif financier
**Méthode :** pour chaque âge, on calcule :
- le **rendement moyen** (rendements logarithmiques, le standard en finance) ;
- la **volatilité** = écart-type des rendements. Plus elle est élevée, plus l'évolution de la valeur est **incertaine**.

On représente chaque âge dans un plan **risque / rendement**, comme pour des actifs financiers.

In [6]:
rend["log_r"] = np.log1p(rend["rendement"])

ra = (rend[(rend["age"] >= 17) & (rend["age"] <= 36)]
      .groupby("age")["log_r"]
      .agg(rendement_moyen="mean", volatilite="std", nb="count")
      .reset_index())
ra[["rendement_moyen", "volatilite"]] *= 100
ra["groupe"] = pd.cut(ra["age"], bins=[16, 23, 28, 36],
                      labels=["Jeunes (17-23)", "Plateau (24-28)", "Seniors (29-36)"])

fig = px.scatter(ra, x="volatilite", y="rendement_moyen", text="age",
                 color="groupe", size="nb",
                 title="Rendement / risque de la valeur d'un joueur selon l'âge (2016-2025)",
                 labels={"volatilite": "Risque : volatilité annuelle (%)",
                         "rendement_moyen": "Rendement annuel moyen (%)",
                         "groupe": "Phase"})
fig.update_traces(textposition="top center")
fig.add_hline(y=0, line_dash="dash")
fig.show()

print(ra.round(1).to_string(index=False))

 age  rendement_moyen  volatilite    nb          groupe
  17             83.4        97.2   500  Jeunes (17-23)
  18             70.5        87.1  2460  Jeunes (17-23)
  19             63.0        89.7  6214  Jeunes (17-23)
  20             51.8        83.1 11384  Jeunes (17-23)
  21             40.0        76.6 13988  Jeunes (17-23)
  22             30.9        70.4 15101  Jeunes (17-23)
  23             22.0        64.2 15485  Jeunes (17-23)
  24             15.4        58.4 15410 Plateau (24-28)
  25             11.3        54.9 14925 Plateau (24-28)
  26              5.8        51.4 14062 Plateau (24-28)
  27              0.7        48.3 13127 Plateau (24-28)
  28             -4.0        46.6 12157 Plateau (24-28)
  29            -10.4        44.7 10928 Seniors (29-36)
  30            -18.1        42.4  9753 Seniors (29-36)
  31            -25.8        41.4  8527 Seniors (29-36)
  32            -32.4        40.6  7260 Seniors (29-36)
  33            -37.5        40.9  5959 Seniors 

### Lecture
**1. Une relation rendement / risque quasi linéaire.** Plus le joueur est jeune, plus son rendement **et** son risque sont élevés : 17 ans → +83 % de rendement pour 97 % de volatilité ; 27 ans → environ 0 % pour 48 %. On retrouve le principe fondamental de la finance : pas de rendement élevé sans risque élevé.

**2. Une asymétrie de type capital-risque.** À 24 ans, le rendement **médian** est de 0 % mais le rendement **moyen** est de +15 % : la majorité des joueurs stagnent, tandis qu'une minorité de « pépites » voit sa valeur exploser.

**3. Les seniors, des actifs dominés.** Après 29 ans, le rendement devient négatif (−10 % à −43 %) pour un risque toujours élevé (environ 41-44 %). Au sens de Markowitz, ces actifs sont **dominés** : leur achat se justifie par l'apport **sportif**, pas financier.

**4. Rendement par unité de risque (rendement ÷ volatilité)**
| Âge | 17 | 20 | 23 | 25 | 27 |
|---|---|---|---|---|---|
| Ratio | 0,86 | 0,62 | 0,34 | 0,21 | 0,01 |

Même ajusté du risque, **les jeunes joueurs restent l'investissement le plus rentable**.

**5. Un actif très risqué.** Même pour les seniors, la volatilité dépasse 40 %, soit **2 à 5 fois** celle d'un grand indice boursier (généralement 15-20 %).

## 5. Le ratio de Sharpe des joueurs
**Principe :** pour chaque joueur, on calcule son rendement annuel moyen et sa volatilité, puis son **ratio de Sharpe = rendement ÷ volatilité**. Un Sharpe élevé signifie une valeur qui a **fortement progressé de façon régulière**.

**Échantillon :** joueurs avec au moins **5 années** d'historique et une valeur d'au moins **10 M€** (période 2016-2025).

*Simplification : le taux sans risque est fixé à 0.*

In [7]:
rend = rend.sort_values(["player_id", "annee"])

stats = (rend.groupby("player_id")
         .agg(nb_annees=("log_r", "count"),
              rendement_moyen=("log_r", "mean"),
              volatilite=("log_r", "std"),
              valeur_actuelle=("valeur", "last"),
              age=("age", "last"),
              position=("position", "first"))
         .reset_index())

stats = stats[(stats["nb_annees"] >= 5) &
              (stats["valeur_actuelle"] >= 10_000_000) &
              (stats["volatilite"] > 0)].copy()
stats["sharpe"] = stats["rendement_moyen"] / stats["volatilite"]
stats[["rendement_moyen", "volatilite"]] *= 100
stats["valeur_M"] = stats["valeur_actuelle"] / 1e6

noms_joueurs = q("SELECT player_id, name FROM players")
stats = stats.merge(noms_joueurs, on="player_id", how="left")

# Graphique 1 : tous les joueurs dans le plan risque / rendement
fig = px.scatter(stats, x="volatilite", y="rendement_moyen", color="position",
                 size="valeur_M", hover_name="name",
                 hover_data={"age": True, "valeur_M": ":.0f", "sharpe": ":.2f"},
                 title=f"Rendement / risque de {len(stats)} joueurs valant au moins 10 M€",
                 labels={"volatilite": "Volatilité annuelle (%)",
                         "rendement_moyen": "Rendement annuel moyen (%)",
                         "position": "Poste"})
fig.add_hline(y=0, line_dash="dash")
fig.show()

# Graphique 2 : le top 15 des ratios de Sharpe
top = stats.nlargest(15, "sharpe").sort_values("sharpe")
fig = px.bar(top, x="sharpe", y="name", orientation="h", color="position",
             hover_data={"age": True, "valeur_M": ":.0f"},
             title="Top 15 des joueurs par ratio de Sharpe",
             labels={"sharpe": "Ratio de Sharpe", "name": ""})
fig.show()

print(stats.nlargest(15, "sharpe")[["name", "position", "age", "valeur_M",
      "nb_annees", "rendement_moyen", "volatilite", "sharpe"]]
      .round(2).to_string(index=False))

                name   position  age  valeur_M  nb_annees  rendement_moyen  volatilite  sharpe
        Dilane Bakwa     Attack   23      32.0          5            93.39       31.77    2.94
      Angelo Stiller   Midfield   24      45.0          5            62.27       23.64    2.63
  Georges Mikautadze     Attack   25      28.0          5            63.00       25.29    2.49
    Viktor Johansson Goalkeeper   27      10.0          5            73.78       33.25    2.22
    Djordje Petrovic Goalkeeper   26      28.0          7            80.50       38.43    2.09
          Paul Nebel   Midfield   23      18.0          5            62.27       34.02    1.83
     Antoine Semenyo     Attack   25      65.0          6            81.13       45.37    1.79
         Maxi Araújo   Defender   25      22.0          6            64.83       36.88    1.76
Jørgen Strand Larsen     Attack   25      40.0          7            85.59       48.88    1.75
       Malik Tillman   Midfield   23      35.0    

### Lecture
- **Dilane Bakwa** obtient le meilleur ratio de Sharpe (**2,94**) : +93 % par an en moyenne pour seulement 32 % de volatilité, soit une progression à la fois forte et régulière. Il est suivi par **Angelo Stiller** (2,63) et **Georges Mikautadze** (2,49).
- **Deux gardiens** figurent dans le top 5 (Viktor Johansson, Djordje Petrovic) : leur valeur progresse moins vite, mais de façon plus régulière.
- **Tous les joueurs du top 15 ont entre 23 et 27 ans**, ce qui confirme l'analyse par âge.
- Le nuage de points confirme à nouveau la relation **rendement / risque** : les joueurs les plus volatils sont aussi ceux qui ont le plus progressé.

### ⚠️ Limites
**Biais de sélection (look-ahead bias).** On a sélectionné les joueurs valant **aujourd'hui** au moins 10 M€, donc uniquement ceux qui ont réussi. C'est pourquoi presque tous les rendements sont positifs. Ce classement mesure **qui a le mieux progressé**, mais il ne dit **pas** qui il fallait acheter. Un vrai test d'investissement doit sélectionner les joueurs **au moment de l'achat**, sans connaître la suite (section suivante).

**Échantillon court.** La volatilité est calculée sur seulement 5 à 7 années par joueur : les ratios de Sharpe sont donc des estimations fragiles.

## 6. Backtest : acheter tous les joueurs à plus de 10 M€ fin 2020
**Objectif :** corriger le biais de sélection de la section précédente. On sélectionne les joueurs **au moment de l'investissement** (valeur ≥ 10 M€ fin 2020), sans connaître leur avenir, puis on mesure leur valeur fin 2025.

**Indicateurs par tranche d'âge (âge en 2020) :**
- **Performance du portefeuille** : valeur totale 2025 ÷ valeur totale 2020 − 1
- **Performance médiane** d'un joueur
- **% de gagnants** : joueurs dont la valeur a augmenté
- **% de sortis** : joueurs sans valorisation en 2025 (retraite, départ vers un championnat non suivi)

*Hypothèse prudente : un joueur sorti du marché est compté avec une valeur nulle dans le portefeuille.*

In [8]:
bt = q("""
WITH annuel AS (
    SELECT player_id, YEAR(date) AS annee,
           arg_max(market_value_in_eur, date) AS valeur
    FROM player_valuations
    WHERE market_value_in_eur > 0
    GROUP BY 1, 2
)
SELECT a.player_id, p.name, p.position,
       2020 - YEAR(TRY_CAST(p.date_of_birth AS DATE)) AS age_2020,
       a.valeur AS v2020,
       b.valeur AS v2025
FROM annuel a
LEFT JOIN annuel b ON a.player_id = b.player_id AND b.annee = 2025
JOIN players p     ON a.player_id = p.player_id
WHERE a.annee = 2020
  AND a.valeur >= 10000000
  AND p.date_of_birth IS NOT NULL
""")

bt["sorti"] = bt["v2025"].isna()
bt["perf_5ans"] = bt["v2025"] / bt["v2020"] - 1
bt["tranche_age"] = pd.cut(bt["age_2020"], bins=[0, 21, 24, 27, 30, 45],
                           labels=["21 ans et -", "22-24", "25-27", "28-30", "31 et +"])

res = (bt.groupby("tranche_age", observed=True)
         .agg(nb=("player_id", "count"),
              pct_sortis=("sorti", lambda s: s.mean() * 100),
              perf_mediane=("perf_5ans", lambda s: s.median() * 100),
              pct_gagnants=("perf_5ans", lambda s: (s.dropna() > 0).mean() * 100),
              valeur_2020_M=("v2020", lambda s: s.sum() / 1e6),
              valeur_2025_M=("v2025", lambda s: s.sum() / 1e6))
         .reset_index())
res["perf_portefeuille"] = (res["valeur_2025_M"] / res["valeur_2020_M"] - 1) * 100

fig = px.bar(res, x="tranche_age", y="perf_portefeuille",
             color=np.where(res["perf_portefeuille"] >= 0, "Gain", "Perte"),
             color_discrete_map={"Gain": "#00CC96", "Perte": "#EF553B"},
             text=res["perf_portefeuille"].round(0).astype(int).astype(str) + "%",
             title="Backtest : performance 2020 → 2025 d'un portefeuille de joueurs à plus de 10 M€",
             labels={"tranche_age": "Âge en 2020", "perf_portefeuille": "Performance sur 5 ans (%)",
                     "color": ""})
fig.show()

total_2020 = res["valeur_2020_M"].sum()
total_2025 = res["valeur_2025_M"].sum()
print(f"Portefeuille global : {total_2020:,.0f} M€ → {total_2025:,.0f} M€ "
      f"({(total_2025 / total_2020 - 1):+.1%})")
print(res.round(1).to_string(index=False))

Portefeuille global : 23,781 M€ → 12,756 M€ (-46.4%)
tranche_age  nb  pct_sortis  perf_mediane  pct_gagnants  valeur_2020_M  valeur_2025_M  perf_portefeuille
21 ans et - 154         5.8          16.7          55.9         3638.0         5145.4               41.4
      22-24 290         5.5         -41.2          27.4         7148.0         5252.8              -26.5
      25-27 269         8.9         -78.6           2.4         6788.5         1734.6              -74.4
      28-30 190        24.7         -89.1           0.0         4930.0          521.0              -89.4
    31 et +  71        38.0         -91.3           0.0         1276.5          102.7              -92.0


### Lecture
**Le portefeuille global perd 46 % en 5 ans** : 23,8 Md€ fin 2020 → 12,8 Md€ fin 2025.

| Âge en 2020 | Nb | Performance du portefeuille | Performance médiane | % gagnants | % sortis |
|---|---|---|---|---|---|
| 21 ans et moins | 154 | **+41 %** | +17 % | 56 % | 6 % |
| 22-24 ans | 290 | −27 % | −41 % | 27 % | 6 % |
| 25-27 ans | 269 | −74 % | −79 % | 2 % | 9 % |
| 28-30 ans | 190 | −89 % | −89 % | **0 %** | 25 % |
| 31 ans et plus | 71 | −92 % | −91 % | **0 %** | 38 % |

- **Seule la tranche des 21 ans et moins est rentable.** Aucun joueur de 28 ans ou plus en 2020 n'a pris de la valeur.
- **Asymétrie de type capital-risque :** chez les plus jeunes, le joueur médian gagne +17 %, mais le portefeuille gagne +41 %. Quelques pépites portent la performance.
- La part de joueurs **sortis du marché** augmente avec l'âge (38 % pour les 31 ans et plus).
- Résultat obtenu alors que les valeurs de 2020 étaient déjà réduites par le Covid : on achetait « en bas de marché ».

### Leçon : l'ampleur du biais de sélection
Le classement du Sharpe (section 5) suggérait des rendements presque toujours positifs. En sélectionnant les joueurs **au moment de l'achat**, sans connaître la suite, le portefeuille perd 46 %.

💼 **Conclusion :** un joueur n'est pas une action, c'est un **actif qui se déprécie**, comme une machine ou un véhicule. Il ne prend de la valeur que s'il est acheté **très jeune**.

## 7. Valeur comptable ou valeur de marché : l'amortissement des joueurs
**Principe comptable (IAS 38) :** un transfert est une **immobilisation incorporelle**. Son coût est amorti de façon linéaire sur la durée du contrat. Exemple : 50 M€ sur 5 ans → 10 M€ par an, et la valeur comptable passe de 50 à 0.

**Question :** la valeur de marché d'un joueur baisse-t-elle **plus vite ou moins vite** que sa valeur comptable ?
- Valeur de marché > valeur comptable → **plus-value latente** (le club gagne s'il revend)
- Valeur de marché < valeur comptable → **moins-value latente** (le club perd s'il revend)

**Méthode :** transferts d'au moins 5 M€ réalisés entre 2018 et 2020. On exprime tout en **base 100 = prix payé**, puis on suit la valeur médiane pendant 5 ans, par tranche d'âge au moment de l'achat.

*Hypothèse simplificatrice : contrat de 5 ans pour tous les transferts (la durée réelle n'est pas disponible dans la base). Les joueurs sortis du marché ne sont plus suivis.*

In [9]:
amort = q("""
WITH annuel AS (
    SELECT player_id, YEAR(date) AS annee,
           arg_max(market_value_in_eur, date) AS valeur
    FROM player_valuations
    WHERE market_value_in_eur > 0
    GROUP BY 1, 2
),
achats AS (
    SELECT t.player_id, p.name,
           YEAR(t.transfer_date) AS annee_achat,
           t.transfer_fee AS prix,
           YEAR(t.transfer_date) - YEAR(TRY_CAST(p.date_of_birth AS DATE)) AS age_achat
    FROM transfers t
    JOIN players p ON t.player_id = p.player_id
    WHERE t.transfer_fee >= 5000000
      AND t.transfer_date BETWEEN DATE '2018-01-01' AND DATE '2020-12-31'
      AND p.date_of_birth IS NOT NULL
)
SELECT a.*, an.annee - a.annee_achat AS t, an.valeur
FROM achats a
JOIN annuel an ON a.player_id = an.player_id
WHERE an.annee - a.annee_achat BETWEEN 0 AND 5
""")

amort["indice_marche"] = amort["valeur"] / amort["prix"].astype(float) * 100
amort["tranche"] = pd.cut(amort["age_achat"], bins=[0, 21, 24, 27, 45],
                          labels=["21 ans et -", "22-24", "25-27", "28 et +"])

courbe = (amort.groupby(["tranche", "t"], observed=True)["indice_marche"]
               .median().reset_index())

fig = px.line(courbe, x="t", y="indice_marche", color="tranche", markers=True,
              title="Valeur de marché médiane vs valeur comptable (base 100 = prix payé)",
              labels={"t": "Années après l'achat",
                      "indice_marche": "Indice (100 = prix payé)",
                      "tranche": "Âge à l'achat"})
fig.add_trace(go.Scatter(x=list(range(6)), y=[100 * (1 - t / 5) for t in range(6)],
                         name="Valeur comptable (amortissement 5 ans)",
                         mode="lines+markers", line=dict(dash="dash", color="black")))
fig.update_layout(hovermode="x unified")
fig.show()

# Part des joueurs en plus-value latente au bout de 3 ans (valeur comptable = 40)
t3 = amort[amort["t"] == 3].copy()
t3["plus_value"] = t3["indice_marche"] > 40
print("Nombre de transferts par tranche :")
print(amort[amort["t"] == 0].groupby("tranche", observed=True).size().to_string())
print("\n% de joueurs en plus-value latente après 3 ans :")
print((t3.groupby("tranche", observed=True)["plus_value"].mean() * 100).round(1).to_string())

Nombre de transferts par tranche :
tranche
21 ans et -    208
22-24          331
25-27          261
28 et +        124

% de joueurs en plus-value latente après 3 ans :
tranche
21 ans et -    76.7
22-24          74.9
25-27          66.7
28 et +        33.9


### Lecture
| Âge à l'achat | Nb de transferts | Valeur de marché après 3 ans (base 100) | Valeur comptable après 3 ans | % en plus-value latente |
|---|---|---|---|---|
| 21 ans et moins | 208 | environ 107 | 40 | **76,7 %** |
| 22-24 ans | 331 | environ 76 | 40 | 74,9 % |
| 25-27 ans | 261 | environ 58 | 40 | 66,7 % |
| 28 ans et plus | 124 | environ 29 | 40 | **33,9 %** |

**1. Une prime de potentiel pour les jeunes.** À l'achat, la valeur de marché des 21 ans et moins vaut environ 87 % du prix payé : les clubs paient environ **15 % au-dessus** de la valeur estimée (environ 8 % pour les 22-24 ans).

**2. Des plus-values latentes pour les moins de 28 ans.** La valeur de marché reste au-dessus de la valeur comptable. Pour les 21 ans et moins, elle **augmente** même (environ 114 en année 4) alors que la valeur comptable tend vers 0. C'est le fondement du modèle de **trading de joueurs** : acheter jeune, amortir, revendre avec une plus-value de cession.

**3. Des moins-values latentes pour les 28 ans et plus.** Leur valeur de marché passe **sous** la valeur comptable dès la 1ʳᵉ année et y reste jusqu'à la 4ᵉ. Une revente anticipée génère une perte, et seul 1 joueur sur 3 est en plus-value après 3 ans.

💼 **Lecture d'auditeur :** lorsque la valeur recouvrable d'un actif devient inférieure à sa valeur comptable, la norme **IAS 36** impose un **test de dépréciation**. Pour les joueurs recrutés à 28 ans et plus, ce test conduirait fréquemment à une dépréciation.

*Limites : contrat de 5 ans supposé pour tous ; les joueurs sortis du marché ne sont plus suivis, ce qui rend les courbes probablement trop optimistes, surtout pour les plus âgés.*

---
# ✅ Conclusion du Module 2 — Le joueur vu comme un actif financier

### Les 5 messages clés
1. **Un cycle de vie en 3 phases** : forte appréciation jusqu'à 23 ans, plateau qui s'érode de 24 à 28 ans, puis dépréciation accélérée (jusqu'à −33 % par an après 35 ans).
2. **Une relation rendement / risque classique** : les jeunes joueurs offrent le rendement le plus élevé, avec le risque le plus élevé. Les joueurs de plus de 29 ans sont des **actifs dominés**.
3. **Un actif très volatil** : la volatilité dépasse 40 % à tous les âges, soit 2 à 5 fois celle d'un indice boursier.
4. **Le backtest corrige le biais de sélection** : un portefeuille de joueurs à plus de 10 M€ acheté fin 2020 perd **46 %** en 5 ans. Seuls les 21 ans et moins sont rentables (+41 %).
5. **Valeur comptable et valeur de marché divergent** : l'amortissement linéaire crée des **plus-values latentes** pour les jeunes, et des **moins-values latentes** pour les 28 ans et plus.

### Conclusion
Un joueur n'est pas une action, c'est un **actif qui se déprécie**. Sa valeur ne se crée qu'en l'achetant **jeune**, en assumant un risque élevé, avec une logique proche du **capital-risque** : beaucoup de paris, quelques grandes réussites.

➡️ **Module suivant :** la rentabilité des transferts et l'efficacité des clubs.